# 📓 실습 02. LangGraph 기반 예외 처리 에이전트 군집 설계

이 실습에서는 반도체 CMP 공정 설비에 이상 징후 센서 신호가 수신되었을 때, 지식 매뉴얼 검색(GraphRAG)을 거쳐 최종 정비 조치 권고안을 엔지니어에게 발송 승인받는 순환 구조 에이전트 워크플로우를 상태 기계(State Machine) 클래스로 시뮬레이션 구현합니다.

In [ ]:
import time

# 1. 에이전트 상태 구조 정의
class AgentState:
    def __init__(self, sensor_value, drift_detected=False):
        self.state = {
            "sensor_value": sensor_value,
            "drift_detected": drift_detected,
            "diagnostic_result": "",
            "search_sop": "",
            "approval_status": "PENDING"
        }

In [ ]:
# 2. 상태 기계 노드 및 조건부 분기(Conditional Routing) 로직 구현
class CMPAgentSwarm:
    def __init__(self):
        pass
        
    def node_diagnostic(self, state):
        print("[Diagnostic Node] CMP 공정 센서 데이터 분석 중...")
        val = state.state["sensor_value"]
        if val <= 80:
            state.state["diagnostic_result"] = "NOZZLE_CLOGGED"
        else:
            state.state["diagnostic_result"] = "NORMAL"
        return state
        
    def node_graph_search(self, state):
        print("[GraphRAG Node] 관련 SOP 조치 지침서 탐색 중...")
        res = state.state["diagnostic_result"]
        if res == "NOZZLE_CLOGGED":
            state.state["search_sop"] = "[SOP-01] 초순수로 노즐을 1.5 bar 압력으로 역방향 플러싱하십시오."
        else:
            state.state["search_sop"] = "모니터링 유지"
        return state
        
    def node_approval(self, state):
        print("[Approval Node] 정비 조치 승인 대기 중...")
        # 가상 엔지니어 승인 시뮬레이션
        state.state["approval_status"] = "APPROVED"
        return state
        
    def run_workflow(self, sensor_value):
        state = AgentState(sensor_value)
        
        # 그래프 흐름 제어
        state = self.node_diagnostic(state)
        
        # 조건부 라우팅 분기
        if state.state["diagnostic_result"] == "NOZZLE_CLOGGED":
            state = self.node_graph_search(state)
            state = self.node_approval(state)
            print(f"\n🎉 [워크플로우 결과] 고장 조치 가이드 발행 완료: {state.state['search_sop']} (승인 상태: {state.state['approval_status']})")
        else:
            print("\n✅ [워크플로우 결과] 장비 정상 작동 중. 모니터링 유지.")
            
swarm = CMPAgentSwarm()
print("=== [테스트 1: 노즐 막힘 징후 센서 수신 (75 mL/min)] ===")
swarm.run_workflow(75)

print("\n=== [테스트 2: 정상 센서 수신 (120 mL/min)] ===")
swarm.run_workflow(120)